# MIT-BIH ECG Signal Exploration

Phase 1 EDA before building the streaming pipeline.
Understand the signal structure, annotation labels, and arrhythmia distribution.

**Run `python scripts/download_mitdb.py` first.**

In [ ]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
DATA_DIR = Path('../data/raw')
print('Setup complete.')

: 

## 1. Load a Record

In [ ]:
RECORD = '100'
record = wfdb.rdrecord(str(DATA_DIR / RECORD))
ann    = wfdb.rdann(str(DATA_DIR / RECORD), 'atr')

print(f'Record     : {RECORD}')
print(f'Channels   : {record.sig_name}')
print(f'Sampling   : {record.fs} Hz')
print(f'Duration   : {record.sig_len / record.fs / 60:.1f} minutes')
print(f'Samples    : {record.sig_len:,}')
print(f'Annotations: {len(ann.symbol)}')

## 2. Plot 10 Seconds of ECG Signal

In [ ]:
fs      = record.fs
start_s = 0
end_s   = 10
start   = start_s * fs
end     = end_s   * fs

signal  = record.p_signal[start:end, 0]  # MLII lead
t       = np.arange(len(signal)) / fs

# Get annotations in this window
ann_mask   = (ann.sample >= start) & (ann.sample < end)
ann_samples = ann.sample[ann_mask] - start
ann_symbols = np.array(ann.symbol)[ann_mask]

fig, ax = plt.subplots()
ax.plot(t, signal, color='steelblue', linewidth=0.8, label='MLII lead')

for s, sym in zip(ann_samples, ann_symbols):
    color = 'red' if sym == 'V' else 'orange' if sym == 'A' else 'green'
    ax.axvline(s / fs, color=color, alpha=0.5, linewidth=1.2)
    ax.text(s / fs, signal.max() * 0.9, sym, fontsize=8, color=color, ha='center')

ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title(f'Record {RECORD} — 10 seconds of ECG (MLII lead)')
patches = [
    mpatches.Patch(color='green',  label='Normal (N)'),
    mpatches.Patch(color='red',    label='Ventricular (V)'),
    mpatches.Patch(color='orange', label='Atrial (A)'),
]
ax.legend(handles=patches, loc='upper right')
plt.tight_layout()
plt.show()

## 3. Annotation Distribution Across All Records

In [ ]:
records_to_check = ['100','101','102','103','104','105','106','107','108','109']
all_annotations  = []

for rec in records_to_check:
    try:
        a = wfdb.rdann(str(DATA_DIR / rec), 'atr')
        for sym in a.symbol:
            all_annotations.append({'record': rec, 'annotation': sym})
    except Exception:
        print(f'Record {rec} not found — skipping.')

df_ann = pd.DataFrame(all_annotations)
counts = df_ann['annotation'].value_counts().reset_index()
counts.columns = ['annotation', 'count']

ANNOTATION_MAP = {
    'N': 'Normal', 'L': 'LBBB', 'R': 'RBBB',
    'A': 'Atrial premature', 'V': 'Ventricular premature',
    'F': 'Fusion', '/': 'Paced', '~': 'Signal quality',
    'Q': 'Unclassifiable', 'f': 'Fusion paced', '!': 'Ventricular flutter',
}
counts['label'] = counts['annotation'].map(lambda x: ANNOTATION_MAP.get(x, x))

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=counts.head(12), x='annotation', y='count', ax=ax, palette='viridis')
ax.set_title('Annotation Distribution — 10 MIT-BIH Records')
ax.set_xlabel('Beat Annotation')
ax.set_ylabel('Count')
for bar, label in zip(ax.patches, counts['label'].head(12)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            label, ha='center', va='bottom', fontsize=8, rotation=20)
plt.tight_layout()
plt.show()

print(counts.to_string(index=False))

## 4. Heart Rate Estimation (Sliding Window Preview)

In [ ]:
# Simulate what the Spark window aggregation does
# Count beats per 30-second window, compute heart rate

fs          = record.fs
window_s    = 30
window_samp = window_s * fs
total_s     = record.sig_len // fs

beat_samples = ann.sample[np.isin(ann.symbol, list('NLRAVFJ'))]
hr_records   = []

for t_start in range(0, total_s - window_s, 5):  # 5-second slide
    t_end    = t_start + window_s
    s_start  = t_start * fs
    s_end    = t_end   * fs
    beats_in = np.sum((beat_samples >= s_start) & (beat_samples < s_end))
    hr_bpm   = beats_in * (60 / window_s)
    hr_records.append({'time_s': t_start, 'heart_rate_bpm': hr_bpm})

df_hr = pd.DataFrame(hr_records)

fig, ax = plt.subplots()
ax.plot(df_hr['time_s'] / 60, df_hr['heart_rate_bpm'], color='steelblue', linewidth=0.9)
ax.axhline(60,  color='orange', linestyle='--', linewidth=1, label='Bradycardia threshold (60 bpm)')
ax.axhline(100, color='red',    linestyle='--', linewidth=1, label='Tachycardia threshold (100 bpm)')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title(f'Record {RECORD} — Estimated Heart Rate (30s sliding window, 5s slide)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean HR : {df_hr['heart_rate_bpm'].mean():.1f} bpm")
print(f"Min HR  : {df_hr['heart_rate_bpm'].min():.1f} bpm")
print(f"Max HR  : {df_hr['heart_rate_bpm'].max():.1f} bpm")

## 5. Schema Design Preview

This is what each Kafka message will look like from the producer.

In [ ]:
import json
from datetime import datetime, timezone

sample_i  = 650  # Example: an annotated beat
ann_idx   = dict(zip(ann.sample, ann.symbol))
ann_sym   = ann_idx.get(sample_i, '')

example_message = {
    'patient_id':       RECORD,
    'device_id':        f'wearable_{RECORD}',
    'timestamp':        datetime.now(timezone.utc).isoformat(),
    'sample_index':     sample_i,
    'channel_0':        float(record.p_signal[sample_i, 0]),
    'channel_1':        float(record.p_signal[sample_i, 1]),
    'sampling_rate':    int(record.fs),
    'annotation':       ann_sym,
    'annotation_label': {'N': 'Normal beat', 'V': 'Premature ventricular contraction'}.get(ann_sym, ''),
    'is_arrhythmia':    ann_sym in {'V', 'A', 'E', 'F', 'a', 'J', 'S'},
}

print(json.dumps(example_message, indent=2))